# ML Workstop 

## The Scenario (00:00 - 05:00)

**The Threat:** Unidentified small uncrewed aerial systems (sUAS) are encroaching on friendly airspace. Radar operators are overwhelmed.

**The Goal:** Deploy a machine learning model to classify incoming radar tracks as either Threat (1) or Non-Threat (0).

**Commander's Intent:** "We cannot afford fratricide or collateral damage. I accept the risk of occasionally missing a small drone (False Negative), but under no circumstances will this system trigger defensive fire on a friendly or neutral aircraft (False Positive)."

## Phase 1: Problem & Data (05:00 - 15:00)

In your Jupyter Notebook, you will first generate the synthetic radar dataset.

**Task:** Review the features. Are they sufficient? Could there be inherent bias in how these radar profiles were historically labeled?

In [ ]:
# Only run this cell after downloading and selecting your kernel
!python.exe -m pip install --upgrade pip
!pip install pandas numpy matplotlib scikit-learn seaborn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# 1. Generate Synthetic Radar Data
# Features: Altitude (m), Speed (knots), Radar Cross Section (m^2), RF Emitting (1/0)
X, y = make_classification(n_samples=1000, n_features=4, n_informative=3, n_redundant=0,
                           weights=[0.7, 0.3], flip_y=0.05, random_state=42)

feature_names = ["Altitude", "Speed", "Cross_Section", "RF_Emitting"]
df = pd.DataFrame(X, columns=feature_names)
df['Is_Threat'] = y

print("Radar Data Snapshot:")
print(df.head())

# Split into Training (for the AI) and Testing (for our evaluation)
X_train, X_test, y_train, y_test = train_test_split(df[feature_names], df['Is_Threat'], test_size=0.2, random_state=42)



## Phase 2: Model Selection & Training (15:00 - 25:00)

You must choose the underlying mathematical framework for the AI.

**Logistic Regression:** Simple, highly explainable, but might underfit complex evasive maneuvers.

**Random Forest:** Powerful, handles non-linear data well, but can overfit.

**Multi-Layer Perceptron (Neural Network):** Highly capable, but acts as a "black box."

**Task:** Change the CHOSEN_MODEL variable in your notebook. Run the cell to train the model instantly.



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# --- STUDENT CHOICE ---
# Options: 'LOG_REG', 'RANDOM_FOREST', 'NEURAL_NET'
CHOSEN_MODEL = 'NEURAL_NET' 
# ----------------------

print(f"Initializing and Training: {CHOSEN_MODEL}...")

if CHOSEN_MODEL == 'LOG_REG':
    model = LogisticRegression()
elif CHOSEN_MODEL == 'RANDOM_FOREST':
    model = RandomForestClassifier(max_depth=5, random_state=42)
elif CHOSEN_MODEL == 'NEURAL_NET':
    model = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=1000, random_state=42)
    

# Train the model
model.fit(X_train, y_train)
print("Training Complete. Model is ready for evaluation.")



## Phase 3: Evaluation & Commander's Intent (25:00 - 40:00)

The model is trained, but is it safe? We must look at the Confusion Matrix.

**Task:** Evaluate the matrix. Based on the Commander's Intent (Zero False Positives), is standard accuracy enough, or do we need to manually adjust the decision threshold to prioritize Precision?


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Get raw probability scores
probabilities = model.predict_proba(X_test)[:, 1]

# --- STUDENT CHOICE ---
# Adjust the probability threshold required to fire (Default is 0.50). 
# How high must it be to satisfy the Commander's Zero-False-Positive intent?
FIRE_THRESHOLD = 0.50

# ----------------------

# Apply the custom threshold
predictions = (probabilities >= FIRE_THRESHOLD).astype(int)

# Generate Confusion Matrix
cm = confusion_matrix(y_test, predictions)
print("Classification Report:\n", classification_report(y_test, predictions))

# Plot
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Neutral', 'Predicted Threat'],
            yticklabels=['Actual Neutral', 'Actual Threat'])
plt.title(f"Confusion Matrix (Threshold: {FIRE_THRESHOLD})")
plt.show()


## Phase 4: Interpretation & Ethics Debrief (40:00 - 50:00)

If the model makes a lethal recommendation, human operators must know why. We will use Explainable AI (SHAP) to crack open the model.

**Task:** Run the SHAP explainer on a specific radar track. Which feature (Speed, Altitude, etc.) was the driving factor for the AI classifying it as a threat? 

If the model relies entirely on RF_Emitting, is it vulnerable to Adversarial Evasion?


In [ ]:
import shap

print("Generating SHAP values for tactical explanation...")

# Initialize SHAP explainer (using a subset of data for speed)
explainer = shap.Explainer(model.predict, X_test[:50])
shap_values = explainer(X_test[:50])

# Visualize the reasoning for Target #0
print("\n--- XAI Report: Target #0 ---")
shap.plots.waterfall(shap_values[0])


## Final Debriefing Questions (Class Discussion)

**The Threshold Tradeoff:** When you raised the threshold to 0.85 (or higher) to eliminate False Positives, what happened to your False Negatives (Missed Threats)?

**Algorithmic Bias:** If this model was trained exclusively on daytime, clear-weather radar profiles, what ethical and tactical failures might occur during a night-time deployment?

**The Final Call:** As the AI Systems Integration Officer, do you sign the authorization to deploy this system tonight? Why or why not?
